# HASTIKA Task B -- overnight sweep

**Sidebar, before you run:** Accelerator `GPU T4 x2` or `GPU P100`, Internet **on**.
Then **Save Version -> Save & Run All**. Do not use an interactive session: this runs
for hours and an interactive tab dies with the browser.

This notebook ranks seven ideas on a 15% holdout, promotes the winners to full 5-fold
runs, blends them, and packages **a CodaBench zip for every arm that produced
predictions**. Read `RESULTS.md` in the Output tab tomorrow and submit the top row.

It stops itself. `BUDGET_HOURS` is a wall clock: no new GPU work starts once the
remaining time cannot fit the next stage, so nothing is cut off mid-fold.


## 0. Settings


In [ ]:
REPO   = "https://github.com/robinpnalex/Hastika-ICON2026.git"
BRANCH = "task-b"

BUDGET_HOURS = 10.5   # Kaggle kills the session at 12 h. This leaves room for the
                      # clone, the installs, and the model downloads.
RESERVE_MIN  = 20     # held back for blending, decoding and zipping, all CPU-only
STAGE2_MAX   = 4      # how many holdout winners get promoted to a 5-fold run
SEEDS        = "42"   # "42 43 44" averages three runs and costs 3x
EPOCHS       = 6


## 1. Clone and install


In [ ]:
import os, subprocess, sys, pathlib

WORK = "/kaggle/working/hastika"
if os.path.isdir(WORK + "/.git"):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, "--depth", "1", REPO, WORK],
                   check=True)
os.chdir(WORK)
os.environ["PYTHONPATH"] = os.path.join(WORK, "src")
pathlib.Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
print("cwd:", os.getcwd())


In [ ]:
# sentencepiece and protobuf are what mDeBERTa's tokenizer needs; without them
# that arm dies at load with an unhelpful message.
subprocess.run('pip install -q emoji ftfy sentencepiece protobuf "transformers>=4.45,<6"',
               shell=True, check=True)

import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "no GPU -- set Accelerator in the sidebar"
print("gpu:", torch.cuda.get_device_name(0),
      "| bf16:", torch.cuda.is_bf16_supported(), "(fp16 is used when this is False)")


## 2. Smoke test

Two minutes, and it fails loudly. You are checking that the device line names your GPU
and that the run reaches `wrote .../predictions.csv`. The score here is noise.


In [ ]:
subprocess.run([sys.executable, "-u", "-m", "hastika.task_b.train", "--tag", "b_smoke",
                "--folds", "0", "--epochs", "1", "--limit", "300"], check=True)


## 3. The sweep

Everything from here is unattended. What it runs, in priority order, and why each one
is here rather than being a learning rate:

| arm | idea | why it might work |
|---|---|---|
| `b_tapt` | masked-LM pass over in-domain text | MuRIL spends 2.28 wordpieces per word on this register |
| `b_xlmr` | XLM-R instead of MuRIL | the corpus is 0% Kannada script, so MuRIL's Indic vocab is dead weight |
| `b_mdeberta` | mDeBERTa-v3 | same bet, strongest multilingual base encoder |
| `b_hing` | HingRoBERTa | trained on romanized Hindi-English social media, the nearest public register |
| `b_focal` | focal loss | stops spending capacity on the Gender mass it already gets right |
| `b_rdrop` | R-Drop | at 3,143 rows the binding constraint is variance, not capacity |
| `b_large` | MuRIL-large | more capacity, three times the time, runs only if the budget survives |

Then the top `STAGE2_MAX` get a 5-fold run, `ensemble.py` weight-searches a blend over
all of them plus the TF-IDF floor, and `decode_b.py` fits a macro-F1-optimal decision
rule on each one's out-of-fold predictions.

Every stage writes its zip as soon as it has predictions, and `RESULTS.md` is rewritten
after every arm. If this dies at hour nine, everything it finished is still there.


In [ ]:
cmd = [sys.executable, "-u", "-m", "experiments.task_b.overnight",
       "--budget-hours", str(BUDGET_HOURS),
       "--reserve-min", str(RESERVE_MIN),
       "--stage2-max", str(STAGE2_MAX),
       "--seeds", SEEDS,
       "--epochs", str(EPOCHS),
       "--out", "/kaggle/working"]
print(" ".join(cmd), flush=True)
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                     text=True, bufsize=1)
for line in p.stdout:
    sys.stdout.write(line)
p.wait()
print("sweep exit", p.returncode)


## 4. What to submit


In [ ]:
print(pathlib.Path("/kaggle/working/RESULTS.md").read_text())

print("\nzips ready to upload to CodaBench:")
for z in sorted(pathlib.Path("/kaggle/working/subs").glob("*.zip")):
    print(" ", z.name, f"{z.stat().st_size/1024:.1f} KB")


## 5. Getting it out

Open the version's **Output** tab. Download `RESULTS.md` and whichever zip its top row
names, then upload that zip to the Task B phase on CodaBench.

Read the table this way. **Five-fold rows beat holdout rows at equal score**, because a
holdout arm trained on 15% less data and is measured on 472 rows rather than 3,143. A
gap under about one point is inside seed noise at this corpus size, so prefer the
simpler arm when two are that close.

Only `/kaggle/working` reaches the Output tab. Model weights cache elsewhere and do not
count against the 20 GB cap.
